<a href="https://colab.research.google.com/github/Sujitha519/AgriMatch_AI/blob/main/PyQuest_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio

import ast
import random
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import gradio as gr

# --- 1. DATA & LEVELS ---
LEVELS_DATA = [
    {"lvl": 1, "title": "Level 1: Variables", "prompt": "Create variable 'x' equal to 10 and print it.", "keyword": "x", "min_len": 8, "hints": ["Use '=' to set x to 10", "Use print(x)"]},
    {"lvl": 2, "title": "Level 2: Strings", "prompt": "Create string variable 'hero' set to 'PyQuest' and print upper case.", "keyword": "upper", "min_len": 12, "hints": ["Use .upper() method", "print(hero.upper())"]},
    {"lvl": 3, "title": "Level 3: Conditional Logic", "prompt": "Write an if condition checking if x > 5 and print 'Big'.", "keyword": "if", "min_len": 10, "hints": ["Structure: if x > 5:", "Inside block: print('Big')"]},
    {"lvl": 4, "title": "Level 4: List Basics", "prompt": "Create a list named 'inventory' with 3 items.", "keyword": "[", "min_len": 12, "hints": ["Use square brackets []", "Separate items with commas"]},
    {"lvl": 5, "title": "Level 5: For Loops", "prompt": "Write a for loop printing numbers 0 to 4 using range().", "keyword": "range", "min_len": 15, "hints": ["Use for i in range(5):", "Indent print(i) inside loop"]},
    {"lvl": 6, "title": "Level 6: While Loops", "prompt": "Create a while loop running while count < 3.", "keyword": "while", "min_len": 15, "hints": ["Initialize count = 0 first", "Don't forget count += 1 inside"]},
    {"lvl": 7, "title": "Level 7: Functions", "prompt": "Define a function 'cast_spell' that returns 'Boom'.", "keyword": "def", "min_len": 18, "hints": ["Use 'def cast_spell():'", "Use 'return \"Boom\"'"]},
    {"lvl": 8, "title": "Level 8: Default Parameters", "prompt": "Define 'greet' function with default parameter name='Hero'.", "keyword": "name=", "min_len": 20, "hints": ["def greet(name='Hero'):", "return f'Hello {name}'"]},
    {"lvl": 9, "title": "Level 9: Dictionaries", "prompt": "Create a dict 'stats' with key 'hp' equal to 100.", "keyword": "hp", "min_len": 10, "hints": ["Use curly braces {}", "stats = {'hp': 100}"]},
    {"lvl": 10, "title": "⚡ Level 10: BOSS BATTLE - Dragon Defeater", "prompt": "Write a function 'attack_dragon(hp, dmg)' returning hp - dmg.", "keyword": "def", "min_len": 25, "hints": ["Define attack_dragon with 2 args", "Return hp minus dmg"]},
    {"lvl": 11, "title": "Level 11: List Comprehension", "prompt": "Create a list of squares for 0 to 4 in one line.", "keyword": "for", "min_len": 20, "hints": ["[x**2 for x in range(5)]", "Wrap in square brackets"]},
    {"lvl": 12, "title": "Level 12: Lambda Expressions", "prompt": "Create a lambda function 'double' multiplying x by 2.", "keyword": "lambda", "min_len": 15, "hints": ["double = lambda x: x * 2", "No return keyword needed"]},
    {"lvl": 13, "title": "Level 13: Error Handling", "prompt": "Write a try/except block catching ZeroDivisionError.", "keyword": "try", "min_len": 20, "hints": ["Start with try:", "Catch with except ZeroDivisionError:"]},
    {"lvl": 14, "title": "Level 14: File I/O Simulation", "prompt": "Use 'with open' syntax to open 'log.txt' in write mode.", "keyword": "open", "min_len": 20, "hints": ["with open('log.txt', 'w') as f:", "f.write('done')"]},
    {"lvl": 15, "title": "Level 15: Modules & Imports", "prompt": "Import math module and calculate square root of 16.", "keyword": "import", "min_len": 15, "hints": ["import math", "math.sqrt(16)"]},
    {"lvl": 16, "title": "Level 16: OOP - Classes", "prompt": "Create a class 'Player' with an __init__ setting name.", "keyword": "class", "min_len": 25, "hints": ["class Player:", "def __init__(self, name):"]},
    {"lvl": 17, "title": "Level 17: OOP - Inheritance", "prompt": "Create class 'Mage' inheriting from class 'Player'.", "keyword": "Player", "min_len": 20, "hints": ["class Mage(Player):", "pass or add unique methods"]},
    {"lvl": 18, "title": "Level 18: Decorators", "prompt": "Define a basic wrapper function decorator.", "keyword": "@", "min_len": 30, "hints": ["Use @my_decorator syntax", "Return inner wrapper function"]},
    {"lvl": 19, "title": "Level 19: Generators", "prompt": "Create a generator function yield_items using 'yield'.", "keyword": "yield", "min_len": 20, "hints": ["Use yield instead of return", "Loop and yield items"]},
    {"lvl": 20, "title": "🔥 Level 20: FINAL BOSS - Dark Overlord AI", "prompt": "Write a multi-class system with abstract methods & exception handling.", "keyword": "class", "min_len": 40, "hints": ["Combine Classes + Exceptions", "Build a complete mini-system"]}
]

# --- 2. GAME STATE ---
state = {
    "level": 1,
    "xp": 0,
    "streak": 1,
    "hp": 100,
    "hints_used": {i: 0 for i in range(1, 21)},
    "rank": "Novice Coder 🐣"
}

RANKS = [
    (0, "Novice Coder 🐣"),
    (200, "Script Kiddie 💻"),
    (500, "Python Apprentice 📜"),
    (1000, "Code Ninja 🥷"),
    (1800, "Syntax Master 🔮"),
    (3000, "AI Overlord 🤖"),
    (5000, "Godlike Developer ⚡")
]

# --- 3. LOGIC FUNCTIONS ---
def update_rank():
    for threshold, rname in reversed(RANKS):
        if state["xp"] >= threshold:
            state["rank"] = rname
            break

def evaluate_code(code_str, level_num):
    target = LEVELS_DATA[level_num - 1]
    if len(code_str.strip()) < target["min_len"]:
        return False, "⚠️ Code is too short or incomplete. Expand your solution!"
    if target["keyword"] not in code_str:
        return False, f"❌ Missing required keyword/symbol: '{target['keyword']}'"
    try:
        ast.parse(code_str)
    except SyntaxError as e:
        return False, f"⚠️ Syntax Error: {e.msg} on line {e.lineno}"
    return True, "✨ AST Syntax Verified! ML Quality Checks Passed."

def get_hud():
    pct = int((state["level"] / 20) * 100)
    bar = "█" * (pct // 5) + "░" * (20 - (pct // 5))
    return f"""
    ### 🎮 PLAYER PROFILE
    * **Rank:** {state['rank']}
    * **Level:** {state['level']} / 20  `[{bar}] {pct}%`
    * **Total XP:** {state['xp']} XP
    * **Daily Streak:** 🔥 {state['streak']} Days
    * **Health:** ❤️ {state['hp']}%
    """

def get_quest_info(lvl_num):
    lvl = LEVELS_DATA[lvl_num - 1]
    is_boss = "BOSS" in lvl["title"]
    prefix = "⚔️ **BOSS BATTLE** ⚔️\n" if is_boss else ""
    return f"{prefix}### {lvl['title']}\n**Mission:** {lvl['prompt']}"

def submit_solution(code, current_lvl_idx):
    lvl_num = int(current_lvl_idx)
    valid, msg = evaluate_code(code, lvl_num)

    if valid:
        is_boss = "BOSS" in LEVELS_DATA[lvl_num - 1]["title"]
        earned_xp = 150 if is_boss else 75
        state["xp"] += earned_xp
        if lvl_num == state["level"] and state["level"] < 20:
            state["level"] += 1
        update_rank()
        feedback = f"🎉 **QUEST CLEARED!** +{earned_xp} XP!\n{msg}"
    else:
        state["hp"] = max(10, state["hp"] - 5)
        feedback = f"💥 **ATTACK FAILED!** -5 HP\n{msg}"

    return feedback, get_hud(), get_quest_info(state["level"]), state["level"]

def request_hint(lvl_num):
    lvl_idx = int(lvl_num) - 1
    hints = LEVELS_DATA[lvl_idx]["hints"]
    used = state["hints_used"][lvl_idx + 1]

    if used < len(hints):
        hint_txt = hints[used]
        state["hints_used"][lvl_idx + 1] += 1
        state["xp"] = max(0, state["xp"] - 15)
        update_rank()
        return f"💡 **AI Hint:** {hint_txt} (-15 XP Penalty)", get_hud()
    return "💡 No remaining hints for this quest!", get_hud()

def spin_daily_wheel():
    prizes = [25, 50, 100, 200, "🔥 Streak Doubler (+100 XP)"]
    prize = random.choice(prizes)
    if isinstance(prize, int):
        state["xp"] += prize
        res = f"🎰 **WHEEL SPIN:** You won +{prize} Bonus XP!"
    else:
        state["xp"] += 100
        state["streak"] += 1
        res = f"🎰 **JACKPOT!** {prize}!"
    update_rank()
    return res, get_hud()

def load_level_select(target_lvl):
    return get_quest_info(int(target_lvl)), int(target_lvl)

# --- 4. GRADIO UI ---
with gr.Blocks(theme=gr.themes.Monochrome(), title="PyQuest AI") as demo:
    gr.Markdown("# ⚡ PyQuest AI: 20-Level Arcade Coding Quest")

    with gr.Row():
        with gr.Column(scale=2):
            quest_display = gr.Markdown(get_quest_info(1))
            code_editor = gr.Code(
                value="# Write your Python code here\nx = 10\nprint(x)",
                language="python",
                label="PyQuest Terminal v2.0",
                lines=8
            )
            with gr.Row():
                btn_submit = gr.Button("🚀 Execute & Attack", variant="primary")
                btn_hint = gr.Button("💡 AI Hint Console")
            console_output = gr.Markdown("```\nSystem Ready. Awaiting Code Execution...\n```")

        with gr.Column(scale=1):
            hud_display = gr.Markdown(get_hud())
            with gr.Accordion("🗺️ Quest Map (Level Selector)", open=True):
                level_selector = gr.Slider(minimum=1, maximum=20, value=1, step=1, label="Jump to Unlocked Level")
                btn_jump = gr.Button("🎯 Select Level")
            with gr.Accordion("🎰 Daily Rewards", open=False):
                btn_spin = gr.Button("🎡 Spin Daily Wheel")
                wheel_output = gr.Markdown("")
            gr.Markdown("### 🏆 Global Leaderboard")
            gr.Dataframe(
                value=pd.DataFrame([
                    ["1", "ByteSlayer ⚡", "4,850 XP", "Godlike"],
                    ["2", "PyMaster 🧙‍♂️", "3,200 XP", "AI Overlord"],
                    ["3", "CodeNinja 🥷", "1,950 XP", "Syntax Master"],
                    ["4", "You (Player)", f"{state['xp']} XP", state['rank']]
                ], columns=["Rank", "Player", "Score", "Tier"]),
                interactive=False
            )

    active_lvl_state = gr.Number(value=1, visible=False)

    btn_submit.click(
        fn=submit_solution,
        inputs=[code_editor, active_lvl_state],
        outputs=[console_output, hud_display, quest_display, active_lvl_state]
    )
    btn_hint.click(
        fn=request_hint,
        inputs=[active_lvl_state],
        outputs=[console_output, hud_display]
    )
    btn_spin.click(
        fn=spin_daily_wheel,
        outputs=[wheel_output, hud_display]
    )
    btn_jump.click(
        fn=load_level_select,
        inputs=[level_selector],
        outputs=[quest_display, active_lvl_state]
    )

demo.launch(share=True, debug=True)

KeyboardInterrupt: 